# 01 - Exploratory data analysis

Monthly revenue, products, countries and customer purchasing patterns (Phase 2).

This notebook reads the outputs in `results/` (produced by the scripts in `src/`); it does not repeat the cleaning logic.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
RES = ROOT / "results"
pd.set_option("display.width", 200)

def csv(name):
    path = RES / name
    if not path.exists():
        print(f"{name} not found: run `python src/run_all.py` from the repository root first.")
        return None
    return pd.read_csv(path)

def js(name):
    path = RES / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


## Data after cleaning

In [ ]:
funnel = csv('cleaning_funnel.csv'); display(funnel)
eda = js('eda_summary.json')
{k: eda.get(k) for k in ['cleaned_rows','unique_customers','unique_invoices','countries','date_min','date_max','total_gross_revenue_gbp','partial_months']}

## Monthly revenue (partial months marked)

In [ ]:
m = csv('monthly_revenue.csv')
if m is not None:
    full, part = m[~m.is_partial_month], m[m.is_partial_month]
    plt.figure(figsize=(12,4.5)); plt.plot(full.Month, full.revenue_gbp, marker='o', label='full months')
    plt.scatter(part.Month, part.revenue_gbp, color='red', marker='x', s=70, label='partial month')
    plt.xticks(rotation=60, ha='right'); plt.ylabel('Revenue (GBP)'); plt.legend(); plt.tight_layout(); plt.show()

## Top products and countries

In [ ]:
display(csv('top_10_products.csv')); display(csv('country_analysis.csv').head(10))

## Customer purchasing patterns

In [ ]:
display(pd.Series(eda.get('purchasing_patterns', {})).to_frame('value'))
display(csv('purchase_frequency_buckets.csv'))
new = csv('monthly_new_vs_returning.csv')
if new is not None:
    new.set_index('month')[['returning_customers','new_customers']].plot(kind='bar', stacked=True, figsize=(12,4)); plt.ylabel('Active customers'); plt.tight_layout(); plt.show()

## Outliers (diagnostic only; nothing removed)

In [ ]:
display(csv('outlier_summary.csv'))